In [ ]:
!pip install open_clip_torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
from pathlib import Path
import ast
import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import open_clip
from PIL import Image

In [ ]:
IS_COLAB = True

DATA_DIR = Path('/content/drive/MyDrive/Projects/SnuAiChallenge/Data') if IS_COLAB else Path('../data')
DATASET_DIR = DATA_DIR / 'snuaichallenge_data'

CKPT_PATH = DATA_DIR / 'ckpt'
REPORT_PATH = DATA_DIR / 'Reports'

CKPT_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
if not DATASET_DIR.exists():
    import kagglehub

    kagglehub.login()
    kagglehub.competition_download("snuaichallenge", output_dir=DATA_DIR)

# Pre-processing

In [ ]:
_, clip_train_image_preprocess, clip_test_image_preprocess = \
    open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')

In [ ]:
from typing import Literal


class SnuAiChallengeDataset(Dataset):
    """
    SNU AI Challenge dataset.

    Attributes
    ----------
    metadata : list[tuple]
        A list of rows with ["Id", "Input_1", "Input_2", "Input_3", "Input_4", "Sentence"] information.
    """
    default_transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    def __init__(self, path: Path, variant: Literal['train', 'test'], transform=None):
        self.path = path
        self.variant = variant
        self.transform = transform if transform is not None else self.default_transform

        self._load_metadata()
        self.cls2idx = {
            (1, 2, 3, 4): 0,
            (1, 2, 4, 3): 1,
            (1, 3, 2, 4): 2,
            (1, 3, 4, 2): 3,
            (1, 4, 2, 3): 4,
            (1, 4, 3, 2): 5,
            (2, 1, 3, 4): 6,
            (2, 1, 4, 3): 7,
            (2, 3, 1, 4): 8,
            (2, 3, 4, 1): 9,
            (2, 4, 1, 3): 10,
            (2, 4, 3, 1): 11,
            (3, 1, 2, 4): 12,
            (3, 1, 4, 2): 13,
            (3, 2, 1, 4): 14,
            (3, 2, 4, 1): 15,
            (3, 4, 1, 2): 16,
            (3, 4, 2, 1): 17,
            (4, 1, 2, 3): 18,
            (4, 1, 3, 2): 19,
            (4, 2, 1, 3): 20,
            (4, 2, 3, 1): 21,
            (4, 3, 1, 2): 22,
            (4, 3, 2, 1): 23,
        }

    def _load_metadata(self):
        self.metadata = []

        with open(self.path / f'{self.variant}.csv', newline='', encoding='utf-8') as f:
            csv_reader = csv.reader(f)
            next(csv_reader)  # skip header

            for row in csv_reader:
                self.metadata.append(tuple(row))

    def __getitem__(self, idx: int):
        image_id, image_1, image_2, image_3, image_4, sentence, *etc = self.metadata[idx]

        images = [
            Image.open(self.path / self.variant / image_id / image_filename).convert('RGB')
            for image_filename in [image_1, image_2, image_3, image_4]
        ]
        images = [self.transform(image) for image in images]
        images = torch.stack(images)

        if self.variant == 'train':
            answer, _is_no_ordering = etc

            label = self.cls2idx[tuple(ast.literal_eval(answer))]
            label = torch.tensor(label).long()

            return images, sentence, label

        return images, sentence

    def __len__(self):
        return len(self.metadata)

In [ ]:
train_dataset = SnuAiChallengeDataset(DATASET_DIR, 'train', transform=clip_train_image_preprocess)
test_dataset = SnuAiChallengeDataset(DATASET_DIR, 'test', transform=clip_test_image_preprocess)

# Model

In [ ]:
hps = {
    'experiment_number': 1,
    'n_epochs': 1,
    'batch_size': 32,
    'lr': 1e-3,
}

In [ ]:
class O4FFusionTransformer(nn.Module):
    def __init__(self, text_encoder, frames_encoder, tokenizer, frames_dim=512, text_dim=512, hidden_dim=64,  num_classes=24, num_heads=8, num_fusion_layers=2, dropout=0.1):
        super().__init__()

        self.text_encoder = text_encoder
        self.frames_encoder = frames_encoder
        self.tokenizer = tokenizer

        self.frames_proj = nn.Linear(frames_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        transformer_encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim * 4, dropout=dropout, batch_first=True, activation='gelu')
        self.fusion = nn.TransformerEncoder(transformer_encoder_layer, num_layers=num_fusion_layers)

        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, frames, sentences):
        """
        frames: (B, 4, C, H, W)
        """
        input_ids = self.tokenizer(list(sentences)).to(frames.device)
        text_feats = self.text_encoder(input_ids)
        text_feats /= text_feats.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        text_feats = self.text_proj(text_feats).unsqueeze(1)

        frame_feats = self.frames_encoder(frames.view(frames.shape[0] * frames.shape[1], *frames.shape[2:]))
        frame_feats /= frame_feats.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        frame_feats = self.frames_proj(frame_feats)
        frame_feats = frame_feats.view(frames.shape[0], frames.shape[1], frame_feats.shape[-1])

        cls = self.cls_token.expand(input_ids.size(0), 1, -1)
        x = torch.cat([
            cls,
            text_feats,
            frame_feats[:, 0:1, :],
            frame_feats[:, 1:2, :],
            frame_feats[:, 2:3, :],
            frame_feats[:, 3:4, :],
        ], dim=1)

        x = self.fusion(x)

        logits = self.head(x[:, 0, :])

        return logits


def build_default_o4ffusiontransformer(freeze_baseline=True, device='cpu'):
    clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    clip_model.eval().to(device)

    if freeze_baseline:
        for param in clip_model.parameters():
            param.requires_grad = False

    model = O4FFusionTransformer(clip_model.encode_text, clip_model.encode_image, tokenizer)
    return model

In [ ]:
generator = torch.Generator().manual_seed(RANDOM_SEED)

train_dataloader = DataLoader(train_dataset, shuffle=True, generator=generator, batch_size=hps['batch_size'], num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=hps['batch_size'], num_workers=2, pin_memory=True)

In [ ]:
model = build_default_o4ffusiontransformer(device=DEVICE)
model.to(DEVICE)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=hps['lr'])

In [ ]:
sum([param.numel() for param in model.parameters()])

In [ ]:
n_epochs = hps['n_epochs']
experiment_number = hps['experiment_number']
all_losses, losses_per_epoch = [], []

for epoch in range(1, n_epochs + 1):
    loss_accum = 0

    for batch_number, (frames, sentences, label) in enumerate(train_dataloader, start=1):
        optimizer.zero_grad(set_to_none=True)

        frames = frames.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)

        logits = model(frames, sentences)

        loss = loss_fn(logits, label)

        loss_accum += loss.item()
        all_losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        print(f'Batch #: [{batch_number}/{len(train_dataloader)}] Loss: [{loss.item()}]')

    loss_per_epoch = loss_accum / len(train_dataloader)
    losses_per_epoch.append(loss_per_epoch)

    torch.save(model.state_dict(), CKPT_PATH / f'o4f-fusion-transformer_{experiment_number}_e{epoch}.pt')

    print(f'Epoch: [{epoch}/{n_epochs}] Loss: [{loss_per_epoch}]')